# RFM Recalculation - Quick Start Example

This notebook demonstrates how to:
1. Load the RFM calculation utilities
2. Calculate RFM scores for a dataframe
3. Analyze the results
4. Update the database

## Prerequisites
- PostgreSQL database running
- Required packages: pandas, sqlalchemy, psycopg2-binary, numpy

## Setup - Import Dependencies

In [ ]:
import sys
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# Add scripts directory to path
scripts_dir = Path('../scripts')
sys.path.insert(0, str(scripts_dir))

# Import RFM utilities
from rfm_utils import RFMCalculator, RFM_CONFIG, create_rfm_report

print("✓ Dependencies imported successfully")

## Database Configuration

In [ ]:
from sqlalchemy import create_engine, inspect, text

# Database connection
DB_CONFIG = {
    "host": "172.18.0.1",
    "database": "postgres",
    "user": "postgres",
    "password": "postgres",
    "port": 5441,
}

# Create engine
engine = create_engine(
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}@"
    f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

# Test connection
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("✓ Database connection successful")
except Exception as e:
    print(f"✗ Connection failed: {e}")

## List Available Tables

In [ ]:
# Get inspector
inspector = inspect(engine)
all_tables = inspector.get_table_names()

# Filter for user_segment tables
user_segment_tables = [t for t in all_tables if t.startswith('user_segment_')]

print(f"Found {len(user_segment_tables)} user_segment tables:")
for i, table in enumerate(sorted(user_segment_tables), 1):
    print(f"  {i}. {table}")

## Load Sample Data

In [ ]:
# Load a sample table
table_name = 'user_segment_agglomerative'  # Change this to your table

query = f'SELECT * FROM "{table_name}" LIMIT 100'  # Load first 100 rows for testing
df = pd.read_sql(query, engine)

print(f"Loaded {len(df)} rows from {table_name}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

## Initialize RFM Calculator

In [ ]:
# Create calculator with default config
calculator = RFMCalculator()

# Or customize the config
# custom_config = {
#     'r_column': 'days_since_last_event',
#     'r_thresholds': [7, 30],      # More strict
#     'r_scores': [2, 1, 0],
#     'f_column': 'total_purchases',
#     'f_thresholds': [5, 20],      # Higher thresholds
#     'f_scores': [0, 1, 2],
#     'm_column': 'total_spent',
#     'm_thresholds': [100, 500],   # Higher thresholds
#     'm_scores': [0, 1, 2],
#     'segment_thresholds': [0, 2, 3, 4, 6],
# }
# calculator = RFMCalculator(custom_config)

print("✓ RFM Calculator initialized")

## Calculate RFM Scores

In [ ]:
# Calculate RFM
df_with_rfm = calculator.calculate_rfm(df)

print("✓ RFM scores calculated")
print(f"\nNew columns added: recency, frequency, monetary, segment, processed_at")

# Display the updated dataframe
print("\nSample results:")
display_cols = ['user_id', 'days_since_last_event', 'recency', 
               'total_purchases', 'frequency',
               'total_spent', 'monetary', 'segment']
available_cols = [c for c in display_cols if c in df_with_rfm.columns]
print(df_with_rfm[available_cols].head(10))

## Analyze RFM Distribution

In [ ]:
# Recency distribution
print("Recency Score Distribution:")
print(df_with_rfm['recency'].value_counts().sort_index())

# Frequency distribution
print("\nFrequency Score Distribution:")
print(df_with_rfm['frequency'].value_counts().sort_index())

# Monetary distribution
print("\nMonetary Score Distribution:")
print(df_with_rfm['monetary'].value_counts().sort_index())

# Segment distribution
print("\nSegment Distribution:")
segment_dist = df_with_rfm['segment'].value_counts().sort_index()
for seg, count in segment_dist.items():
    pct = (count / len(df_with_rfm) * 100)
    interpretation = calculator.get_segment_interpretation(seg)
    print(f"  Segment {seg}: {count:>3} ({pct:>5.1f}%) - {interpretation}")

## Segment Summary Statistics

In [ ]:
# Get segment summary
summary = calculator.get_segment_summary(df_with_rfm)
print("Segment Summary Statistics:")
print(summary)

## Generate Report

In [ ]:
# Create and display report
report = create_rfm_report(df_with_rfm, calculator)
print(report)

## Preview Changes (Dry Run)

In [ ]:
# Show what would be updated
print(f"\nWould update {len(df_with_rfm)} rows in table '{table_name}'")
print(f"New columns: recency, frequency, monetary, segment, processed_at")

# Show sample of updated data
print("\nSample of updated data:")
cols_to_show = ['user_id', 'recency', 'frequency', 'monetary', 'segment', 'processed_at']
available = [c for c in cols_to_show if c in df_with_rfm.columns]
print(df_with_rfm[available].head(10).to_string())

## Update Database (Optional)

⚠️ **CAUTION**: Uncomment and run the cell below only when you're ready to update the database.
This will replace the existing table with the new RFM scores.

In [ ]:
# UNCOMMENT TO UPDATE THE DATABASE
# WARNING: This will replace the existing table!

# if_exists_option = 'replace'  # Change to 'replace' to overwrite
# 
# try:
#     df_with_rfm.to_sql(
#         table_name,
#         engine,
#         if_exists=if_exists_option,
#         index=False
#     )
#     print(f"✓ Successfully updated {len(df_with_rfm)} rows in '{table_name}'")
# except Exception as e:
#     print(f"✗ Error updating database: {e}")

## Verify Changes (After Update)

In [ ]:
# After updating, verify the changes
# Uncomment to run after database update

# query = f'SELECT recency, frequency, monetary, segment, processed_at FROM "{table_name}" LIMIT 10'
# df_verify = pd.read_sql(query, engine)
# 
# print(f"Verification - First 10 rows from updated table '{table_name}':")
# print(df_verify)
# 
# print(f"\nProcessed at: {df_verify['processed_at'].max()}")

## Recalculate All Tables (Batch Operation)

Process multiple tables in one go.

In [ ]:
# Process all tables
def batch_process_rfm(engine, calculator, tables_to_process=None, dry_run=True):
    """
    Process multiple tables and calculate RFM scores.
    """
    if tables_to_process is None:
        inspector = inspect(engine)
        all_tables = inspector.get_table_names()
        tables_to_process = [t for t in all_tables if t.startswith('user_segment_')]
    
    results = {}
    
    for table_name in tables_to_process:
        print(f"\n► Processing: {table_name}")
        
        try:
            # Load table
            df = pd.read_sql(f'SELECT * FROM "{table_name}"', engine)
            print(f"  Loaded {len(df)} rows")
            
            # Calculate RFM
            df_rfm = calculator.calculate_rfm(df)
            
            # Update database
            if not dry_run:
                df_rfm.to_sql(table_name, engine, if_exists='replace', index=False)
                print(f"  ✓ Updated {len(df_rfm)} rows")
            else:
                print(f"  [DRY RUN] Would update {len(df_rfm)} rows")
            
            results[table_name] = {
                'status': 'success',
                'rows': len(df),
                'segments': df_rfm['segment'].value_counts().to_dict()
            }
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
            results[table_name] = {'status': 'error', 'error': str(e)}
    
    print(f"\n{'='*60}")
    print(f"Batch processing complete: {sum(1 for r in results.values() if r['status'] == 'success')}/{len(results)} successful")
    
    return results

# Run batch processing (dry_run=True by default)
results = batch_process_rfm(engine, calculator, dry_run=True)

# Display summary
print("\nSummary:")
for table, result in results.items():
    if result['status'] == 'success':
        print(f"  {table}: {result['rows']} rows, segments: {result['segments']}")
    else:
        print(f"  {table}: ERROR - {result['error']}")